# Multimodal Fusion Pipeline - Audio + Video Anomaly Detection
Combines Isolation Forest (audio) and Convolutional Autoencoder (video) predictions using weighted late fusion.

In [1]:
import os, numpy as np, cv2, librosa, joblib, warnings
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
print('All libraries imported!')

All libraries imported!


## 1. Load Trained Models

In [111]:
BASE = r'd:\multimodel-surveillence'

# Audio model
audio_model = joblib.load(os.path.join(BASE, 'audio_isolation_forest.pkl'))
audio_scaler = joblib.load(os.path.join(BASE, 'audio_scaler.pkl'))
print('\u2705 Audio Isolation Forest loaded')

# Video model - use compile=False to avoid Keras 3 deserialization issues
video_model = tf.keras.models.load_model(os.path.join(BASE, 'video_transfer_model.h5'), compile=False)
video_model.compile(optimizer='adam', loss='mse')  # Recompile manually
print('\u2705 Video Conv Autoencoder loaded')
video_model.summary()

✅ Audio Isolation Forest loaded
✅ Video Conv Autoencoder loaded


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,618,945 (9.99 MB)

 Trainable params: 1,567,041 (5.98 MB)

 Non-trainable params: 1,051,904 (4.01 MB)

## 2. Define Feature Extraction Functions

In [112]:
IMG_SIZE = 96

def extract_audio_features(file_path, n_mfcc=40):
    y, sr = librosa.load(file_path, duration=30)
    feat = []
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    feat.extend(np.mean(mfccs.T, axis=0)); feat.extend(np.std(mfccs.T, axis=0))
    sc = librosa.feature.spectral_contrast(y=y, sr=sr)
    feat.extend(np.mean(sc.T, axis=0)); feat.extend(np.std(sc.T, axis=0))
    ch = librosa.feature.chroma_stft(y=y, sr=sr)
    feat.extend(np.mean(ch.T, axis=0)); feat.extend(np.std(ch.T, axis=0))
    for fn_name in ['zero_crossing_rate','spectral_centroid','spectral_bandwidth','spectral_rolloff','rms']:
        fn = getattr(librosa.feature, fn_name)
        r = fn(y=y) if fn_name in ['zero_crossing_rate','rms'] else fn(y=y, sr=sr)
        feat.append(np.mean(r)); feat.append(np.std(r))
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128), ref=np.max)
    feat.extend(np.mean(mel.T, axis=0)); feat.extend(np.std(mel.T, axis=0))
    return np.array(feat)

def extract_video_frame(file_path):
    img = cv2.imread(file_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype('float32') / 255.0

print('Feature extraction functions defined.')

Feature extraction functions defined.


## 3. Anomaly Scoring Functions

In [113]:
def get_audio_anomaly_score(audio_path):
    feat = extract_audio_features(audio_path)
    feat_scaled = audio_scaler.transform(feat.reshape(1, -1))
    raw_score = audio_model.decision_function(feat_scaled)[0]
    score = np.clip(-raw_score, -1, 1)
    return (score + 1) / 2

def get_video_anomaly_score(frame):
    inp = frame.reshape(1, IMG_SIZE, IMG_SIZE, 1)
    recon = video_model.predict(inp, verbose=0)
    error = np.mean(np.square(inp - recon))
    score = np.clip(error / 0.1, 0, 1)
    return score, error

print('Scoring functions defined.')

Scoring functions defined.


## 4. Multimodal Fusion Engine

In [114]:
class MultimodalFusion:
    def __init__(self, audio_weight=0.4, video_weight=0.6, threshold=0.2):
        self.audio_weight = audio_weight
        self.video_weight = video_weight
        self.threshold = threshold
    def fuse_scores(self, audio_score, video_score):
        return self.audio_weight * audio_score + self.video_weight * video_score
    def predict(self, audio_score, video_score):
        fused = self.fuse_scores(audio_score, video_score)
        is_anomaly = fused > self.threshold
        return fused, is_anomaly, 'ANOMALY' if is_anomaly else 'NORMAL'
    def predict_audio_only(self, audio_score):
        return audio_score, audio_score > self.threshold, 'ANOMALY' if audio_score > self.threshold else 'NORMAL'
    def predict_video_only(self, video_score):
        return video_score, video_score > self.threshold, 'ANOMALY' if video_score > self.threshold else 'NORMAL'

fusion = MultimodalFusion(audio_weight=0.4, video_weight=0.6, threshold=0.2)
print(f'Fusion Engine: Audio={fusion.audio_weight}, Video={fusion.video_weight}, Threshold={fusion.threshold}')

Fusion Engine: Audio=0.4, Video=0.6, Threshold=0.2


## 5. Test Fusion on Dataset

In [115]:
AUDIO_DIR = os.path.join(BASE, 'DATASET', 'dataset.audio')
VIDEO_DIR = os.path.join(BASE, 'DATASET', 'dataset.videos')

print('Testing Audio Model...')
print(f'{"File":<35} {"Score":<10} {"Prediction"}')
print('-'*60)
audio_results = []
for cat, label in [('normal',0),('anomoly',1)]:
    d = os.path.join(AUDIO_DIR, cat)
    if not os.path.exists(d): continue
    for f in os.listdir(d):
        if not f.endswith('.wav'): continue
        score = get_audio_anomaly_score(os.path.join(d, f))
        pred = 'ANOMALY' if score > 0.2 else 'NORMAL'
        ok = '\u2705' if (label==1 and pred=='ANOMALY') or (label==0 and pred=='NORMAL') else '\u274c'
        print(f'{f:<35} {score:<10.4f} {pred} {ok}')
        audio_results.append({'file':f, 'true':label, 'score':score})

Testing Audio Model...
File                                Score      Prediction
------------------------------------------------------------
man-laughing.wav                    0.4571     ANOMALY ❌
people-in-lounge.wav                0.4864     ANOMALY ❌
spring-weather.wav                  0.4804     ANOMALY ❌
ambulance-siren.wav                 0.4848     ANOMALY ✅
explosion.wav                       0.4908     ANOMALY ✅
glass-shattering.wav                0.4947     ANOMALY ✅


In [116]:
print('\nTesting Video Model on sample frames...')
print(f'{"Category":<15} {"File":<40} {"Error":<12} {"Score":<10} {"Prediction"}')
print('-'*90)
video_results = []
for cat, label in [('normal',0),('anomoly',1)]:
    d = os.path.join(VIDEO_DIR, cat)
    if not os.path.exists(d): continue
    count = 0
    for root, _, files in os.walk(d):
        for f in files:
            if not f.lower().endswith(('.png','.jpg','.jpeg')): continue
            if count >= 10: break
            frame = extract_video_frame(os.path.join(root, f))
            if frame is None: continue
            score, error = get_video_anomaly_score(frame)
            pred = 'ANOMALY' if score > 0.2 else 'NORMAL'
            ok = '\u2705' if (label==1 and pred=='ANOMALY') or (label==0 and pred=='NORMAL') else '\u274c'
            print(f'{cat:<15} {f[:38]:<40} {error:<12.6f} {score:<10.4f} {pred} {ok}')
            video_results.append({'file':f, 'true':label, 'score':score})
            count += 1


Testing Video Model on sample frames...
Category        File                                     Error        Score      Prediction
------------------------------------------------------------------------------------------


ValueError: Exception encountered when calling Functional.call().

[1mInput 0 with name 'None' of layer 'Conv1' is incompatible with the layer: expected axis -1 of input shape to have value 3, but received input with shape (1, 96, 96, 1)[0m

Arguments received by Functional.call():
  • inputs=tf.Tensor(shape=(1, 96, 96, 1), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

## 6. Threshold Optimization

In [ ]:
print('Optimizing thresholds...')
if audio_results:
    a_true = [r['true'] for r in audio_results]
    a_scores = [r['score'] for r in audio_results]
    best_a_th, best_a_f1 = 0.5, 0
    for th in np.arange(0.3, 0.8, 0.05):
        preds = [1 if s>th else 0 for s in a_scores]
        f1 = f1_score(a_true, preds) if sum(preds)>0 else 0
        if f1 > best_a_f1: best_a_f1=f1; best_a_th=th
    print(f'Best audio threshold: {best_a_th:.2f}, F1: {best_a_f1:.4f}')
if video_results:
    v_true = [r['true'] for r in video_results]
    v_scores = [r['score'] for r in video_results]
    best_v_th, best_v_f1 = 0.5, 0
    for th in np.arange(0.1, 0.9, 0.05):
        preds = [1 if s>th else 0 for s in v_scores]
        f1 = f1_score(v_true, preds) if sum(preds)>0 else 0
        if f1 > best_v_f1: best_v_f1=f1; best_v_th=th
    print(f'Best video threshold: {best_v_th:.2f}, F1: {best_v_f1:.4f}')

## 7. Dashboard Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Multimodal Fusion Dashboard', fontsize=16, fontweight='bold')
if audio_results:
    axes[0][0].barh([r['file'][:15] for r in audio_results], [r['score'] for r in audio_results],
                    color=['green' if r['true']==0 else 'red' for r in audio_results], alpha=0.8)
    axes[0][0].axvline(x=0.2, color='black', linestyle='--')
    axes[0][0].set_title('Audio Anomaly Scores')
if video_results:
    axes[0][1].barh([r['file'][:15] for r in video_results], [r['score'] for r in video_results],
                    color=['green' if r['true']==0 else 'red' for r in video_results], alpha=0.8)
    axes[0][1].axvline(x=0.2, color='black', linestyle='--')
    axes[0][1].set_title('Video Anomaly Scores')
axes[1][0].text(0.5,0.7,'Audio (Isolation Forest) ──┐',ha='center',fontsize=11,family='monospace')
axes[1][0].text(0.5,0.5,'                           ├── FUSION ── ANOMALY/NORMAL',ha='center',fontsize=11,family='monospace')
axes[1][0].text(0.5,0.3,'Video (Conv Autoencoder) ──┘',ha='center',fontsize=11,family='monospace')
axes[1][0].axis('off'); axes[1][0].set_title('Architecture')
axes[1][1].pie([fusion.audio_weight, fusion.video_weight], labels=['Audio','Video'],
              autopct='%1.0f%%', colors=['#3498db','#e74c3c'], startangle=90)
axes[1][1].set_title('Fusion Weights')
plt.tight_layout(); plt.show()

## 8. Save Config

In [1]:
import json
config = {'audio_weight': fusion.audio_weight, 'video_weight': fusion.video_weight,
          'threshold': fusion.threshold, 'img_size': IMG_SIZE,
          'audio_model': 'audio_isolation_forest.pkl', 'audio_scaler': 'audio_scaler.pkl',
          'video_model': 'video_transfer_model.h5'}
with open(os.path.join(BASE, 'fusion_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print('Fusion config saved!')
print(json.dumps(config, indent=2))

NameError: name 'fusion' is not defined